In [1]:
# Statistical Validation, Hypothesis Testing, and Business Insights

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Set project paths and create output directories
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy"
)

INPUT_FILE = (
    PROJECT_DIR / "data" / "processed"
    / "European_Bank_Cleaned_Analytical.csv"
)

TABLE_DIR = PROJECT_DIR / "outputs" / "tables" / "statistical_validation"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures" / "statistical_validation"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {INPUT_FILE}")

# ------------------------------------------------------------
# Load and prepare the analytical dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

required_columns = [
    "customerid",
    "exited",
    "isactivemember",
    "numofproducts",
    "balance"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if df.empty:
    raise ValueError("The dataset is empty.")

if df["customerid"].duplicated().any():
    raise ValueError("Duplicate customer IDs detected.")

for col in ["exited", "isactivemember", "numofproducts", "balance"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if df[required_columns[1:]].isna().any().any():
    raise ValueError("Missing or invalid values in required statistical columns.")

if not set(df["exited"].unique()).issubset({0, 1}):
    raise ValueError("Exited must contain only 0 and 1.")

if not set(df["isactivemember"].unique()).issubset({0, 1}):
    raise ValueError("IsActiveMember must contain only 0 and 1.")

if (df["numofproducts"] < 1).any():
    raise ValueError("NumOfProducts contains values below 1.")

print("=" * 75)
print("STATISTICAL VALIDATION AND BUSINESS INSIGHTS")
print("=" * 75)

print(f"\nCustomers loaded: {len(df):,}")
print(f"Columns available: {len(df.columns):,}")

# ------------------------------------------------------------
# Define statistical methodology
# ------------------------------------------------------------

ALPHA = 0.05

methodology = pd.DataFrame([
    {
        "analysis": "Activity versus churn",
        "method": "Chi-square test of independence",
        "null_hypothesis": "Activity status and churn are independent.",
        "alternative_hypothesis": "Activity status and churn are associated.",
        "significance_level": ALPHA
    },
    {
        "analysis": "Product count versus churn",
        "method": "Chi-square test of independence",
        "null_hypothesis": "Product count and churn are independent.",
        "alternative_hypothesis": "Product count and churn are associated.",
        "significance_level": ALPHA
    },
    {
        "analysis": "Balance group versus churn",
        "method": "Chi-square test of independence",
        "null_hypothesis": "Balance group and churn are independent.",
        "alternative_hypothesis": "Balance group and churn are associated.",
        "significance_level": ALPHA
    },
    {
        "analysis": "Balance distribution comparison",
        "method": "Mann-Whitney U test",
        "null_hypothesis": "Balance distributions are equal between churn groups.",
        "alternative_hypothesis": "Balance distributions differ between churn groups.",
        "significance_level": ALPHA
    },
    {
        "analysis": "Relationship tier versus churn",
        "method": "Chi-square test of independence",
        "null_hypothesis": "Relationship tier and churn are independent.",
        "alternative_hypothesis": "Relationship tier and churn are associated.",
        "significance_level": ALPHA
    }
])

methodology.to_csv(
    TABLE_DIR / "statistical_testing_methodology.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Check numerical distributions and statistical assumptions
# ------------------------------------------------------------

numeric_columns = [
    col for col in [
        "age",
        "creditscore",
        "tenure",
        "balance",
        "estimatedsalary",
        "numofproducts"
    ]
    if col in df.columns
]

distribution_results = []

for col in numeric_columns:

    values = df[col].dropna()

    distribution_results.append({
        "variable": col,
        "count": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "standard_deviation": values.std(),
        "minimum": values.min(),
        "maximum": values.max(),
        "skewness": values.skew(),
        "missing_values": df[col].isna().sum()
    })

distribution_summary = pd.DataFrame(distribution_results)

distribution_summary.to_csv(
    TABLE_DIR / "distribution_and_assumption_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nNumerical distribution summary:")
print(distribution_summary.to_string(index=False))

# ------------------------------------------------------------
# Create a reusable categorical association testing function
# ------------------------------------------------------------

statistical_results = []

def categorical_association_test(
    data,
    group_column,
    target_column="exited",
    analysis_name=None
):
    """
    Test association between a categorical variable and churn.

    Uses Fisher's exact test for a 2x2 table when expected
    frequencies are below 5. Otherwise, uses chi-square.
    """

    temp = data[[group_column, target_column]].dropna()

    contingency = pd.crosstab(
        temp[group_column],
        temp[target_column]
    )

    contingency = contingency.reindex(
        columns=[0, 1],
        fill_value=0
    )

    if contingency.shape[0] < 2:
        return None

    if contingency.shape[1] != 2:
        return None

    chi2_stat, chi2_p, degrees_freedom, expected = (
        chi2_contingency(contingency, correction=False)
    )

    minimum_expected = expected.min()

    if (
        contingency.shape == (2, 2)
        and minimum_expected < 5
    ):
        odds_ratio, p_value = fisher_exact(contingency.values)
        test_used = "Fisher Exact Test"
    else:
        p_value = chi2_p
        odds_ratio = np.nan
        test_used = "Chi-square Test"

    total = contingency.to_numpy().sum()

    minimum_dimension = min(
        contingency.shape[0] - 1,
        contingency.shape[1] - 1
    )

    cramers_v = (
        np.sqrt(chi2_stat / (total * minimum_dimension))
        if total > 0 and minimum_dimension > 0
        else np.nan
    )

    result = {
        "analysis": analysis_name or group_column,
        "group_variable": group_column,
        "test_used": test_used,
        "sample_size": total,
        "chi_square_statistic": chi2_stat,
        "degrees_of_freedom": degrees_freedom,
        "p_value": p_value,
        "cramers_v": cramers_v,
        "minimum_expected_frequency": minimum_expected,
        "expected_frequency_below_5": bool(minimum_expected < 5),
        "odds_ratio": odds_ratio,
        "significance_level": ALPHA,
        "statistically_significant": bool(p_value < ALPHA)
    }

    statistical_results.append(result)

    return contingency, result


# ------------------------------------------------------------
# Compare active and inactive customer churn
# ------------------------------------------------------------

df["activity_group"] = df["isactivemember"].map({
    1: "Active",
    0: "Inactive"
})

activity_churn_summary = (
    df.groupby("activity_group")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum())
    )
    .reset_index()
)

activity_churn_summary["churn_rate_pct"] = (
    activity_churn_summary["exited_customers"]
    / activity_churn_summary["total_customers"] * 100
)

activity_churn_summary["retention_rate_pct"] = (
    activity_churn_summary["retained_customers"]
    / activity_churn_summary["total_customers"] * 100
)

activity_test = categorical_association_test(
    df,
    "activity_group",
    analysis_name="Activity Status and Churn"
)

if activity_test is not None:
    activity_contingency, activity_test_result = activity_test

activity_churn_summary.to_csv(
    TABLE_DIR / "activity_churn_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Compare product-count groups and churn
# ------------------------------------------------------------

product_churn_summary = (
    df.groupby("numofproducts")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum())
    )
    .reset_index()
    .sort_values("numofproducts")
)

product_churn_summary["churn_rate_pct"] = (
    product_churn_summary["exited_customers"]
    / product_churn_summary["total_customers"] * 100
)

product_churn_summary["retention_rate_pct"] = (
    product_churn_summary["retained_customers"]
    / product_churn_summary["total_customers"] * 100
)

product_test = categorical_association_test(
    df,
    "numofproducts",
    analysis_name="Product Count and Churn"
)

if product_test is not None:
    product_contingency, product_test_result = product_test

product_churn_summary.to_csv(
    TABLE_DIR / "product_count_churn_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Compare high-balance and lower-balance customer churn
# ------------------------------------------------------------

positive_balances = df.loc[df["balance"] > 0, "balance"]

if positive_balances.empty:
    high_balance_threshold = np.nan
    df["balance_group"] = "Lower Balance"
else:
    high_balance_threshold = positive_balances.quantile(0.75)

    df["balance_group"] = np.where(
        df["balance"] >= high_balance_threshold,
        "High Balance",
        "Lower Balance"
    )

balance_churn_summary = (
    df.groupby("balance_group")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum()),
        average_balance=("balance", "mean"),
        median_balance=("balance", "median")
    )
    .reset_index()
)

balance_churn_summary["churn_rate_pct"] = (
    balance_churn_summary["exited_customers"]
    / balance_churn_summary["total_customers"] * 100
)

balance_churn_summary["retention_rate_pct"] = (
    balance_churn_summary["retained_customers"]
    / balance_churn_summary["total_customers"] * 100
)

balance_test = categorical_association_test(
    df,
    "balance_group",
    analysis_name="Balance Group and Churn"
)

if balance_test is not None:
    balance_contingency, balance_test_result = balance_test

balance_churn_summary.to_csv(
    TABLE_DIR / "balance_group_churn_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Compare continuous balance distributions between churn groups
# ------------------------------------------------------------

retained_balances = df.loc[
    df["exited"] == 0, "balance"
].dropna()

exited_balances = df.loc[
    df["exited"] == 1, "balance"
].dropna()

if len(retained_balances) > 0 and len(exited_balances) > 0:

    mann_whitney_result = mannwhitneyu(
        retained_balances,
        exited_balances,
        alternative="two-sided"
    )

    balance_distribution_result = {
        "analysis": "Balance Distribution: Retained versus Exited",
        "test_used": "Mann-Whitney U Test",
        "retained_sample_size": len(retained_balances),
        "exited_sample_size": len(exited_balances),
        "retained_median_balance": retained_balances.median(),
        "exited_median_balance": exited_balances.median(),
        "u_statistic": mann_whitney_result.statistic,
        "p_value": mann_whitney_result.pvalue,
        "statistically_significant": bool(
            mann_whitney_result.pvalue < ALPHA
        )
    }

else:
    balance_distribution_result = {
        "analysis": "Balance Distribution: Retained versus Exited",
        "test_used": "Mann-Whitney U Test",
        "p_value": np.nan,
        "statistically_significant": False
    }

balance_distribution_df = pd.DataFrame([
    balance_distribution_result
])

balance_distribution_df.to_csv(
    TABLE_DIR / "balance_distribution_mann_whitney_test.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate Wilson confidence intervals for key churn rates
# ------------------------------------------------------------

def calculate_churn_ci(group_name, group_data):

    total = len(group_data)
    exited = int(group_data["exited"].sum())

    if total == 0:
        return {
            "group": group_name,
            "total_customers": 0,
            "exited_customers": 0,
            "churn_rate_pct": np.nan,
            "ci_lower_pct": np.nan,
            "ci_upper_pct": np.nan
        }

    lower, upper = proportion_confint(
        count=exited,
        nobs=total,
        alpha=0.05,
        method="wilson"
    )

    return {
        "group": group_name,
        "total_customers": total,
        "exited_customers": exited,
        "churn_rate_pct": exited / total * 100,
        "ci_lower_pct": lower * 100,
        "ci_upper_pct": upper * 100
    }


confidence_interval_results = []

confidence_interval_results.append(
    calculate_churn_ci("Overall", df)
)

for group_name, group_data in df.groupby("activity_group"):
    confidence_interval_results.append(
        calculate_churn_ci(
            f"Activity: {group_name}",
            group_data
        )
    )

for group_name, group_data in df.groupby("numofproducts"):
    confidence_interval_results.append(
        calculate_churn_ci(
            f"Products: {group_name}",
            group_data
        )
    )

for group_name, group_data in df.groupby("balance_group"):
    confidence_interval_results.append(
        calculate_churn_ci(
            f"Balance: {group_name}",
            group_data
        )
    )

confidence_interval_summary = pd.DataFrame(
    confidence_interval_results
)

confidence_interval_summary.to_csv(
    TABLE_DIR / "churn_rate_confidence_intervals.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Evaluate relationship strength tiers
# ------------------------------------------------------------

df["activity_score"] = df["isactivemember"].astype(float)

df["product_adoption_score"] = (
    np.minimum(df["numofproducts"], 2) / 2
)

df["relationship_strength_index"] = (
    60 * df["activity_score"]
    + 40 * df["product_adoption_score"]
)

rsi_bins = [-0.01, 24, 49, 74, 89, 100]

rsi_labels = [
    "Very Weak",
    "Weak",
    "Moderate",
    "Strong",
    "Very Strong"
]

df["relationship_strength_category"] = pd.cut(
    df["relationship_strength_index"],
    bins=rsi_bins,
    labels=rsi_labels,
    include_lowest=True
)

if df["relationship_strength_category"].isna().any():
    raise ValueError("Some customers were not assigned an RSI tier.")

rsi_summary = (
    df.groupby(
        "relationship_strength_category",
        observed=False
    )
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum()),
        average_rsi=("relationship_strength_index", "mean"),
        average_products=("numofproducts", "mean"),
        active_customers=("isactivemember", "sum")
    )
    .reset_index()
)

rsi_summary["churn_rate_pct"] = (
    rsi_summary["exited_customers"]
    / rsi_summary["total_customers"].replace(0, np.nan) * 100
)

rsi_summary["retention_rate_pct"] = (
    rsi_summary["retained_customers"]
    / rsi_summary["total_customers"].replace(0, np.nan) * 100
)

rsi_test = categorical_association_test(
    df,
    "relationship_strength_category",
    analysis_name="Relationship Strength Tier and Churn"
)

if rsi_test is not None:
    rsi_contingency, rsi_test_result = rsi_test

rsi_summary.to_csv(
    TABLE_DIR / "relationship_strength_churn_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Compare relationship strength across customer segments
# ------------------------------------------------------------

segment_columns = [
    "geography",
    "gender",
    "agegroup",
    "tenuregroup",
    "balancecategory",
    "productcountcategory"
]

available_segment_columns = [
    col for col in segment_columns
    if col in df.columns
]

segment_comparison_results = []

for segment_col in available_segment_columns:

    grouped = (
        df.groupby(segment_col, dropna=False)
        .agg(
            total_customers=("customerid", "count"),
            average_rsi=("relationship_strength_index", "mean"),
            median_rsi=("relationship_strength_index", "median"),
            exited_customers=("exited", "sum")
        )
        .reset_index()
    )

    grouped["churn_rate_pct"] = (
        grouped["exited_customers"]
        / grouped["total_customers"] * 100
    )

    grouped.insert(0, "segment_dimension", segment_col)

    grouped = grouped.rename(
        columns={segment_col: "segment_value"}
    )

    segment_comparison_results.append(grouped)

if segment_comparison_results:

    segment_comparison = pd.concat(
        segment_comparison_results,
        ignore_index=True
    )

else:

    segment_comparison = pd.DataFrame()

segment_comparison.to_csv(
    TABLE_DIR / "relationship_strength_segment_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Build a consolidated statistical test results table
# ------------------------------------------------------------

statistical_results_df = pd.DataFrame(statistical_results)

if not statistical_results_df.empty:

    statistical_results_df["interpretation"] = np.where(
        statistical_results_df["statistically_significant"],
        "Statistically significant association detected.",
        "Insufficient evidence of a statistically significant association."
    )

statistical_results_df.to_csv(
    TABLE_DIR / "statistical_test_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Build an optional explanatory logistic regression model
# ------------------------------------------------------------

model_status = "Not run"

model_metrics = pd.DataFrame()
model_coefficients = pd.DataFrame()
model_confusion_matrix = pd.DataFrame()

try:

    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        average_precision_score,
        confusion_matrix
    )

    # Select original customer characteristics.
    # RSI and its components are excluded to avoid redundant predictors.

    candidate_features = [
        "creditscore",
        "geography",
        "gender",
        "age",
        "tenure",
        "balance",
        "numofproducts",
        "hascrcard",
        "isactivemember",
        "estimatedsalary"
    ]

    model_features = [
        col for col in candidate_features
        if col in df.columns
    ]

    if len(model_features) < 2:
        raise ValueError("Not enough suitable features for logistic regression.")

    model_data = df[model_features + ["exited"]].copy()

    X = model_data[model_features]
    y = model_data["exited"].astype(int)

    if y.nunique() != 2 or y.value_counts().min() < 2:
        raise ValueError("Insufficient target-class observations.")

    numeric_features = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = [
        col for col in X.columns
        if col not in numeric_features
    ]

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessing = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features)
        ]
    )

    logistic_pipeline = Pipeline(steps=[
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    logistic_pipeline.fit(X_train, y_train)

    y_pred = logistic_pipeline.predict(X_test)

    y_probability = logistic_pipeline.predict_proba(X_test)[:, 1]

    model_metrics = pd.DataFrame([{
        "model": "Logistic Regression",
        "training_customers": len(X_train),
        "testing_customers": len(X_test),
        "features_used": ", ".join(model_features),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "f1_score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_test, y_probability),
        "average_precision": average_precision_score(
            y_test, y_probability
        ),
        "class_weight": "balanced",
        "interpretation": (
            "Explanatory baseline using a random stratified holdout; "
            "not a validated future-churn prediction model."
        )
    }])

    model_metrics.to_csv(
        TABLE_DIR / "logistic_regression_model_metrics.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Extract model coefficients for explanatory review

    feature_names = (
        logistic_pipeline
        .named_steps["preprocessing"]
        .get_feature_names_out()
    )

    coefficients = (
        logistic_pipeline
        .named_steps["model"]
        .coef_[0]
    )

    model_coefficients = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "odds_ratio": np.exp(
            np.clip(coefficients, -20, 20)
        )
    })

    model_coefficients["absolute_coefficient"] = (
        model_coefficients["coefficient"].abs()
    )

    model_coefficients = model_coefficients.sort_values(
        "absolute_coefficient",
        ascending=False
    )

    model_coefficients.to_csv(
        TABLE_DIR / "logistic_regression_coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )

    matrix = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    model_confusion_matrix = pd.DataFrame(
        matrix,
        index=["Actual Retained", "Actual Exited"],
        columns=["Predicted Retained", "Predicted Exited"]
    )

    model_confusion_matrix.to_csv(
        TABLE_DIR / "logistic_regression_confusion_matrix.csv",
        encoding="utf-8-sig"
    )

    model_status = "Successfully completed"

except Exception as model_error:

    model_status = f"Not completed: {str(model_error)}"

    print("\nLogistic regression was not completed.")
    print(f"Reason: {model_error}")

# ------------------------------------------------------------
# Identify statistically supported findings
# ------------------------------------------------------------

business_findings = []

for _, result in statistical_results_df.iterrows():

    if result["statistically_significant"]:

        finding = (
            f"{result['analysis']}: statistically significant association "
            f"detected (p={result['p_value']:.6f}, "
            f"Cramer's V={result['cramers_v']:.4f})."
        )

    else:

        finding = (
            f"{result['analysis']}: no statistically significant association "
            f"was established at the 5% significance level "
            f"(p={result['p_value']:.6f})."
        )

    business_findings.append(finding)

# Add practical activity comparison
if len(activity_churn_summary) == 2:

    active_rate = activity_churn_summary.loc[
        activity_churn_summary["activity_group"] == "Active",
        "churn_rate_pct"
    ].iloc[0]

    inactive_rate = activity_churn_summary.loc[
        activity_churn_summary["activity_group"] == "Inactive",
        "churn_rate_pct"
    ].iloc[0]

    business_findings.append(
        f"Observed active-customer churn rate: {active_rate:.2f}%; "
        f"inactive-customer churn rate: {inactive_rate:.2f}%. "
        f"The difference is {inactive_rate - active_rate:.2f} percentage points "
        f"(inactive minus active)."
    )

# Add balance comparison
business_findings.append(
    f"High-balance threshold used: {high_balance_threshold:.2f} "
    f"(75th percentile of positive balances), when defined."
)

business_findings.append(
    "Balance distributions were compared using the Mann-Whitney U test, "
    "which does not require normally distributed balance values."
)

business_findings.append(
    "The RSI analysis evaluates association between a proposed score tier "
    "and observed churn. It does not establish that changing RSI will "
    "cause a change in retention."
)

business_findings.append(
    "The analysis is based on observational customer data. Statistical "
    "associations must not be interpreted as causal effects."
)

business_findings.append(
    "The dataset represents a single-year customer snapshot. It does not "
    "provide a time-based future churn outcome or customer transaction history."
)

business_findings.append(
    f"Logistic regression status: {model_status}."
)

findings_file = TABLE_DIR / "validated_business_insights.txt"

with open(findings_file, "w", encoding="utf-8") as file:

    file.write("STATISTICAL VALIDATION AND BUSINESS INSIGHTS\n")
    file.write("=" * 70 + "\n\n")

    file.write("STATISTICAL METHODOLOGY\n")
    file.write(
        "Significance level: 0.05\n"
        "Categorical associations: Chi-square or Fisher's exact test\n"
        "Association strength: Cramer's V\n"
        "Continuous balance comparison: Mann-Whitney U test\n"
        "Churn confidence intervals: 95% Wilson intervals\n\n"
    )

    file.write("FINDINGS\n")
    file.write("-" * 70 + "\n")

    for finding in business_findings:
        file.write(f"- {finding}\n")

    file.write("\nBUSINESS INTERPRETATION LIMITATIONS\n")
    file.write("-" * 70 + "\n")

    file.write(
        "1. Statistical association does not establish causation.\n"
        "2. P-values should be interpreted alongside effect sizes.\n"
        "3. Small groups may have unstable rates and confidence intervals.\n"
        "4. Multiple statistical tests increase the possibility of false positives.\n"
        "5. The logistic regression model is explanatory and not a validated "
        "future-churn prediction system.\n"
        "6. The dataset does not include transaction-level engagement or "
        "longitudinal customer behaviour.\n"
    )

# ------------------------------------------------------------
# Save statistical analysis customer dataset
# ------------------------------------------------------------

statistical_customer_columns = [
    "customerid",
    "exited",
    "isactivemember",
    "activity_group",
    "numofproducts",
    "balance",
    "balance_group",
    "activity_score",
    "product_adoption_score",
    "relationship_strength_index",
    "relationship_strength_category"
]

for col in [
    "age",
    "geography",
    "gender",
    "tenure",
    "hascrcard",
    "creditscore"
]:
    if col in df.columns:
        statistical_customer_columns.append(col)

df[statistical_customer_columns].to_csv(
    TABLE_DIR / "statistical_validation_customer_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Final validation checks
# ------------------------------------------------------------

validation_checks = []

def add_validation(name, condition, details):
    validation_checks.append({
        "validation_check": name,
        "status": "PASSED" if condition else "FAILED",
        "details": details
    })

add_validation(
    "Customer count reconciliation",
    len(df) == df["customerid"].nunique(),
    "All customer records have unique customer IDs."
)

add_validation(
    "Churn and retention reconciliation",
    int(df["exited"].sum()) + int((df["exited"] == 0).sum()) == len(df),
    "Retained and exited customer counts equal total customers."
)

add_validation(
    "Activity segment reconciliation",
    activity_churn_summary["total_customers"].sum() == len(df),
    "Active and inactive customer counts reconcile."
)

add_validation(
    "Product group reconciliation",
    product_churn_summary["total_customers"].sum() == len(df),
    "Product-count groups reconcile."
)

add_validation(
    "Balance group reconciliation",
    balance_churn_summary["total_customers"].sum() == len(df),
    "Balance groups reconcile."
)

add_validation(
    "RSI tier reconciliation",
    rsi_summary["total_customers"].sum() == len(df),
    "Relationship strength tiers contain all customers."
)

add_validation(
    "RSI score range",
    df["relationship_strength_index"].between(0, 100).all(),
    "All RSI values are between 0 and 100."
)

add_validation(
    "Confidence interval validity",
    (
        confidence_interval_summary["ci_lower_pct"].dropna()
        <= confidence_interval_summary["ci_upper_pct"].dropna()
    ).all(),
    "Wilson lower bounds do not exceed upper bounds."
)

validation_report = pd.DataFrame(validation_checks)

validation_report.to_csv(
    TABLE_DIR / "statistical_validation_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ACTIVITY AND CHURN COMPARISON")
print("=" * 75)

print(activity_churn_summary.to_string(index=False))

print("\n" + "=" * 75)
print("PRODUCT COUNT AND CHURN COMPARISON")
print("=" * 75)

print(product_churn_summary.to_string(index=False))

print("\n" + "=" * 75)
print("BALANCE GROUP AND CHURN COMPARISON")
print("=" * 75)

print(balance_churn_summary.to_string(index=False))

print("\n" + "=" * 75)
print("BALANCE DISTRIBUTION TEST")
print("=" * 75)

print(balance_distribution_df.to_string(index=False))

print("\n" + "=" * 75)
print("STATISTICAL TEST RESULTS")
print("=" * 75)

print(statistical_results_df.to_string(index=False))

print("\n" + "=" * 75)
print("CHURN RATE CONFIDENCE INTERVALS")
print("=" * 75)

print(confidence_interval_summary.to_string(index=False))

print("\n" + "=" * 75)
print("RELATIONSHIP STRENGTH ANALYSIS")
print("=" * 75)

print(rsi_summary.to_string(index=False))

print("\n" + "=" * 75)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 75)

print(f"Model status: {model_status}")

if not model_metrics.empty:
    print(model_metrics.to_string(index=False))

print("\n" + "=" * 75)
print("VALIDATED BUSINESS INSIGHTS")
print("=" * 75)

for finding in business_findings:
    print("-", finding)

print("\n" + "=" * 75)
print("FINAL VALIDATION REPORT")
print("=" * 75)

print(validation_report.to_string(index=False))

failed_checks = validation_report[
    validation_report["status"] == "FAILED"
]

if not failed_checks.empty:
    raise AssertionError(
        "Some validation checks failed. Review the validation report."
    )

print("\nAll statistical validation checks passed successfully.")

print("\n" + "=" * 75)
print("OUTPUT FILES")
print("=" * 75)

for output_file in sorted(TABLE_DIR.glob("*")):
    if output_file.is_file():
        print(output_file.name)

print("\nStatistical validation and business insights completed.")
print(f"Tables and reports saved to: {TABLE_DIR}")
print(f"Figures directory reserved at: {FIGURE_DIR}")

STATISTICAL VALIDATION AND BUSINESS INSIGHTS

Customers loaded: 10,000
Columns available: 26

Numerical distribution summary:
       variable  count          mean     median  standard_deviation  minimum   maximum  skewness  missing_values
            age  10000     38.921800     37.000           10.487806    18.00     92.00  1.011320               0
    creditscore  10000    650.528800    652.000           96.653299   350.00    850.00 -0.071607               0
         tenure  10000      5.012800      5.000            2.892174     0.00     10.00  0.010991               0
        balance  10000  76485.889288  97198.540        62397.405202     0.00 250898.09 -0.141109               0
estimatedsalary  10000 100090.239881 100193.915        57510.492818    11.58 199992.48  0.002085               0
  numofproducts  10000      1.530200      1.000            0.581654     1.00      4.00  0.745568               0

ACTIVITY AND CHURN COMPARISON
activity_group  total_customers  retained_customers 